# Regresión Lógistica

Por qué es importante?

Conseguir un nuevo cliente nuevo representa entre 5x y 25x más que retener uno existente.
Si podemos predecir quien se va, podemos odfecerle un descuento o llamarlo antes de que cancele.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Preprocesamiento de datos, transformar columnas especificas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables categóricas
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluacion para la clasificación
from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt

In [ ]:
# Importar el dataset
df = pd.read_csv("abandono_clientes.csv")

In [ ]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

In [ ]:
# Änálisis del tipo de contrato vs abandono
tabla_contrato = pd.crosstab

### Prepara los datos para el modelo

1. Separar variables predictoras (X) de la variable objetivo
2. Entrenar los datos con (80%) del dataset y probar con el (20%) del dataset

In [ ]:
# Paso 1: Separar variables predictoras (X) y variable objetivo (y)
x = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte",
        "satisfaccion", "contrato"]]
y = df["abandono"]

# Paso 2: Dividir el dataset en conjunto de entrenamiento y prueba
x_train, x_test, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.2,      # 20% de los datos para prueba
    random_state=42,    # Semilla para reproducibilidad 
    stratify=y)
print(f"Datos de entrenamiento: {x_train.shape[0]} filas")
print(f"Datos de prueba: {x_test.shape[0]} filas")
print("-" * 50 )
print(f"Proporción de abandono en entrenamiento:\n{y_train.mean()*100:.1f}%")
print(f"Proporción de abandono en prueba:\n{y_test.mean()*100:.1f}%")
print("-" * 50 )
print("Las proporciones son similares gracias al parametro 'stratify=y' en train_test_split")


In [ ]:
# Definir columnas por tipo de dato
numeric_features = ["antiguedad_meses", "factura_mensual", "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]

# Preprocesamiento de datos
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features), # Dejar las columnas numéricas sin cambios
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)        
    ]
)
# Pipeline para preprocesamiento y modelo
# Garantizar que los datos siempre se procesen antes de llegar al modelo
from sklearn.pipeline import Pipeline
pipe = Pipeline(steps=[ 
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
# Generar predicciones con los daos reales

y_pred = pipe.predict(x_test)

# Mostrar las primeras 10 predicciones vs valores reales

comparacion_pred = pd.DataFrame({
    "Valor Real": y_test.values[:10],
    "Predicción": y_pred[:10],
    "Correcto": ["Si" if r == p else "No" for r, p in zip (y_test.values[:10], y_pred[:10])]
})
print("Comparacion de predicciones vs valores reales:")
print("-" * 50)
print(comparacion_pred)
